In [ ]:
import optuna
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


def objective(trial):

    # Hyperparameters to tune
    n_units = trial.suggest_int("n_units", 32, 256)
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)

    # Model
    model = keras.Sequential([
        layers.Dense(n_units, activation="relu", input_shape=(X_train.shape[1],)),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    # Train
    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=128,
        verbose=0
    )

    # Evaluate
    loss, acc = model.evaluate(X_val, y_val, verbose=0)

    return acc

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best parameters:", study.best_params)